# Disentanglement experiments on Colab

This notebook launches the repository's existing MSP or LibriSpeech trainer. It does not contain a second training implementation. A free runtime is one resumable segment, not a guarantee that a full experiment will finish in one session. Pilot results are development evidence only.

CLUB is used as a probe-architecture-agnostic objective. Its learned variational estimate is not, by itself, proof that every downstream probe fails; use the separate held-out probe phase.

In [ ]:
#@title 1. Experiment selection
DATASET = 'librispeech' #@param ['msp', 'librispeech']
EXPERIMENT = 'libri_grl_stats_gelu' #@param ['msp_baseline','msp_no_pcgrad','msp_no_invariance','msp_soft_routing','msp_no_cross_adversaries','msp_no_adversaries','libri_grl_stats_gelu','libri_club_hybrid','libri_club_pure']
PHASE = 'train' #@param ['train', 'probe']
PROFILE = 'full' #@param ['full', 'pilot']
SEED = 42 #@param {type:'integer'}
SEGMENT_STEPS = 250 #@param {type:'integer'}
MAX_RUNTIME_MINUTES = 600 #@param {type:'integer'}
REPO_URL = 'https://github.com/beimnet777/Final-Proj.git' #@param {type:'string'}
GIT_COMMIT = 'main' #@param {type:'string'}
assert EXPERIMENT.startswith('msp_') == (DATASET == 'msp'), 'Dataset and experiment disagree'

In [ ]:
#@title 2. Mount Drive and clone the pinned source
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os, subprocess
WORK = Path('/content/final-proj')
if not WORK.exists():
    token = ''
    try:
        from google.colab import userdata
        token = userdata.get('GITHUB_TOKEN') or ''
    except Exception:
        pass
    url = REPO_URL.replace('https://', f'https://{token}@') if token else REPO_URL
    subprocess.run(['git', 'clone', url, str(WORK)], check=True)
subprocess.run(['git', '-C', str(WORK), 'fetch', '--all'], check=True)
subprocess.run(['git', '-C', str(WORK), 'checkout', GIT_COMMIT], check=True)
COMMIT = subprocess.check_output(['git','-C',str(WORK),'rev-parse','HEAD'], text=True).strip()
print('Pinned commit:', COMMIT)

In [ ]:
#@title 3. Install dependencies without replacing Colab PyTorch
import sys, subprocess, torch
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(WORK/'Disentanglement/requirements-colab.txt')], check=True)
print('torch', torch.__version__, 'cuda', torch.version.cuda, 'available', torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(p.name, round(p.total_memory/2**30, 1), 'GiB', 'bf16', torch.cuda.is_bf16_supported())
subprocess.run(['df','-h','/content'])

In [ ]:
#@title 4. Download the complete project LibriSpeech dataset before testing
import shutil, subprocess, sys
DRIVE_ROOT = Path('/content/drive/MyDrive/FinalProjColab')
DATA_ROOT = Path('/content/data')
if DATA_ROOT.exists(): shutil.rmtree(DATA_ROOT)
if DATASET == 'librispeech':
    # The complete dataset used in this project: 100 h training plus clean dev/test.
    DATA_ROOT.mkdir(parents=True, exist_ok=True)
    drive_cache = DRIVE_ROOT/'assets'/'openslr'; drive_cache.mkdir(parents=True, exist_ok=True)
    for name in ('train-clean-100', 'dev-clean', 'test-clean'):
        cached_archive = drive_cache/f'{name}.tar.gz'
        local_archive = Path('/content')/f'{name}.tar.gz'
        if cached_archive.exists():
            print(f'Using cached Drive archive: {cached_archive.name}')
            shutil.copy2(cached_archive, local_archive)
        else:
            print(f'Downloading {name} from the official OpenSLR release ...')
            subprocess.run(['wget', '-c', '-O', str(local_archive), f'https://www.openslr.org/resources/12/{name}.tar.gz'], check=True)
            shutil.copy2(local_archive, cached_archive)
        subprocess.run(['tar', '-xzf', str(local_archive), '-C', str(DATA_ROOT)], check=True)
    shutil.copy2(WORK/'Probing/data/librispeech-lexicon.txt', DATA_ROOT/'librispeech-lexicon.txt')
    assert (DATA_ROOT/'LibriSpeech/train-clean-100').exists()
    assert (DATA_ROOT/'LibriSpeech/dev-clean').exists()
    assert (DATA_ROOT/'LibriSpeech/test-clean').exists()
else:
    asset_name = f'msp_{PROFILE}.tar.gz'
    source_archive = DRIVE_ROOT/'assets'/asset_name
    if not source_archive.exists():
        raise FileNotFoundError(f'MSP is licensed and cannot be downloaded automatically. Upload: {source_archive}')
    local_archive = Path('/content')/asset_name
    shutil.copy2(source_archive, local_archive)
    subprocess.run([sys.executable, '-m', 'Disentanglement.colab_bundle', 'verify', '--archive', str(local_archive), '--extract-to', str(DATA_ROOT)], cwd=WORK, check=True)
print('Data ready:', DATA_ROOT)

In [ ]:
#@title 5. Restore the Hugging Face cache locally (optional)
HF_HOME = Path('/content/hf')
cache_archive = DRIVE_ROOT/'assets'/'spear_cache.tar.gz'
if cache_archive.exists() and not HF_HOME.exists():
    subprocess.run(['tar','-xzf',str(cache_archive),'-C','/content'], check=True)
HF_HOME.mkdir(exist_ok=True)
os.environ['HF_HOME'] = str(HF_HOME)
os.environ['HF_HUB_CACHE'] = str(HF_HOME/'hub')

In [ ]:
#@title 6. Resolve and inspect the command (safe dry run)
RUN_NAME = f'{EXPERIMENT}_s{SEED}'
LOCAL_RUN = Path('/content/runs')/RUN_NAME
DRIVE_RUN = DRIVE_ROOT/'runs'/EXPERIMENT/f'seed-{SEED}'
LOCAL_RUN.mkdir(parents=True, exist_ok=True); DRIVE_RUN.mkdir(parents=True, exist_ok=True)
base = [sys.executable, '-m', 'Disentanglement.experiment_runner', '--experiment', EXPERIMENT, '--data_root', str(DATA_ROOT), '--profile', PROFILE, '--phase', PHASE, '--seed', str(SEED), '--output_dir', str(LOCAL_RUN), '--drive_mirror', str(DRIVE_RUN), '--segment_steps', str(SEGMENT_STEPS), '--max_runtime_minutes', str(MAX_RUNTIME_MINUTES)]
if PHASE == 'train': base += ['--effective_batch_size','16','--microbatch_size','auto','--resume','auto','--resume_every','50','--precision','auto']
subprocess.run(base + ['--dry_run'], cwd=WORK, check=True) if PHASE == 'train' else print('Probe command:', base)

In [ ]:
#@title 7. Restore checkpoint and run one segment
for name in ('latest-resume.pt','best.pt','final.pt','metrics.jsonl'):
    src, dst = DRIVE_RUN/name, LOCAL_RUN/name
    if src.exists() and not dst.exists(): shutil.copy2(src, dst)
subprocess.run(base, cwd=WORK, check=True)

In [ ]:
#@title 8. Inspect persisted metrics
import json, pandas as pd, matplotlib.pyplot as plt
metrics = LOCAL_RUN/'metrics.jsonl'
if metrics.exists():
    rows = [json.loads(line) for line in metrics.read_text().splitlines() if line.strip()]
    frame = pd.DataFrame(rows); display(frame.tail())
    numeric = [c for c in frame.columns if c not in {'step','split'} and pd.api.types.is_numeric_dtype(frame[c])]
    if numeric: frame.plot(x='step', y=numeric[:8], subplots=True, figsize=(10, 2*min(8,len(numeric)))); plt.tight_layout()
else:
    print('No validation metrics yet; run another segment or reduce checkpoint interval.')